# Genie Space Quality Evaluation

This notebook implements an LLM-powered evaluation chain for assessing the quality of a
Databricks AI/BI Genie space. It retrieves the space configuration via the Databricks
Genie API and scores it against quality criteria.

### Current Evaluation Criteria

| # | Criterion | Scoring Method |
|---|-----------|---------------|
| 1 | Table & Column Metadata | LLM-evaluated, 1–5 scale (presence, clarity, completeness) |
| 2 | Benchmark Question Quantity | Count-based, 1–5 scale (20+ questions = 5) |
| 3 | Benchmark Question Quality | LLM-evaluated, 1–5 scale (relevance, complexity, uniqueness) |

### Scoring Rubric — Table & Column Metadata

| Factor | What it measures |
|--------|-----------------|
| Table descriptions | Is every table accompanied by a clear, informative description? |
| Column descriptions | Do columns have descriptions that explain their meaning and content? |
| Overall completeness | Are there gaps — tables or columns with missing or vague metadata? |

### Scoring Rubric — Benchmark Question Quantity

| Question Count | Score |
|---------------|-------|
| 0 | 1 |
| 5 | 2 |
| 10 | 3 |
| 15 | 4 |
| 20+ | 5 |

### Scoring Rubric — Benchmark Question Quality

| Factor | What it measures |
|--------|-----------------|
| Relevance | Do the questions align with the space's stated purpose and available data sources? |
| Complexity | Are the questions substantive, or are they overly simplistic? |
| Uniqueness | Are the questions distinct, or are there duplicates / near-duplicates? |

In [ ]:
%pip install databricks-sdk databricks-langchain langchain-core -qU

In [ ]:
import json

from databricks.sdk import WorkspaceClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_databricks import ChatDatabricks

## Configuration

Set your Genie space ID and LLM serving endpoint below.

When running inside a Databricks notebook, `WorkspaceClient()` authenticates automatically.
For local execution, set the `DATABRICKS_HOST` and `DATABRICKS_TOKEN` environment variables.

In [ ]:
GENIE_SPACE_ID = "<your-genie-space-id>"
LLM_ENDPOINT = "databricks-meta-llama-3-1-70b-instruct"

## Retrieve Genie Space Configuration

Fetch the full space definition — including benchmark questions, sample questions,
instructions, and data source metadata — from the Databricks Genie API.

The `serialized_space` field is a JSON string containing the complete space configuration.
Benchmark questions live under `benchmarks.questions` within that structure.

In [ ]:
w = WorkspaceClient()

space = w.genie.get_space(
    space_id=GENIE_SPACE_ID,
    include_serialized_space=True,
)

space_config = json.loads(space.serialized_space)

benchmark_questions = space_config.get("benchmarks", {}).get("questions", [])
sample_questions = space_config.get("config", {}).get("sample_questions", [])
data_sources = space_config.get("data_sources", {})
tables = data_sources.get("tables", [])

print(f"Space:  {space.title}")
print(f"Description:  {space.description or 'N/A'}")
print(f"Data source tables:  {len(tables)}")
print(f"Benchmark questions: {len(benchmark_questions)}")
print(f"Sample questions:    {len(sample_questions)}")

if tables:
    print("\n--- Data Sources ---")
    for tbl in tables:
        desc = " ".join(tbl["description"]) if tbl.get("description") else "no description"
        columns = [c["column_name"] for c in tbl.get("column_configs", [])]
        col_str = f' — columns: {", ".join(columns)}' if columns else ""
        print(f"  • {tbl['identifier']} ({desc}){col_str}")

if benchmark_questions:
    print("\n--- Benchmark Questions ---")
    for i, q in enumerate(benchmark_questions, 1):
        question_text = " ".join(q.get("question", []))
        print(f"  {i}. {question_text}")

## Criterion 1 — Table & Column Metadata

Use the LLM to evaluate the **completeness and clarity** of table and column metadata
on a 1–5 scale. Genie relies heavily on table descriptions and column comments to
interpret user questions, so poor metadata directly degrades answer quality.

The model assesses:

1. **Table descriptions** — Is every table accompanied by a clear, informative description?
2. **Column descriptions** — Do columns have descriptions that explain their meaning and content?
3. **Overall completeness** — Are there significant gaps (tables or columns with missing or vague metadata)?

In [ ]:
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)

# --- build a detailed view of table + column metadata for the LLM ---
metadata_details = []
for tbl in tables:
    tbl_id = tbl["identifier"]
    tbl_desc = " ".join(tbl["description"]) if tbl.get("description") else "(no description)"
    col_configs = tbl.get("column_configs", [])
    col_lines = []
    for col in col_configs:
        col_desc = " ".join(col["description"]) if col.get("description") else "(no description)"
        synonyms = ", ".join(col["synonyms"]) if col.get("synonyms") else None
        parts = [f"    - {col['column_name']}: {col_desc}"]
        if synonyms:
            parts[0] += f"  [synonyms: {synonyms}]"
        col_lines.append(parts[0])
    col_block = "\n".join(col_lines) if col_lines else "    (no column configs)"
    metadata_details.append(f"  Table: {tbl_id}\n  Description: {tbl_desc}\n  Columns:\n{col_block}")

metadata_formatted = "\n\n".join(metadata_details) if metadata_details else "  (no tables)"

metadata_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert evaluator of Databricks AI/BI Genie spaces. "
     "You will be given the table and column metadata configured for a "
     "Genie space. Your job is to score the QUALITY of the metadata on "
     "a 1-5 integer scale.\n\n"
     "You MUST respond with valid JSON only — no markdown fences, no "
     "extra text outside the JSON object."),
    ("human", """Score the table and column metadata for this Genie space.

**Space Name:** {space_name}
**Space Description:** {space_description}
**Number of tables:** {num_tables}

**Table & Column Metadata:**
{metadata}

Evaluate the metadata on three factors, each scored 1-5:

1. **table_descriptions** (1-5) — Does every table have a description?
   Is each description clear, specific, and informative enough for an LLM
   to understand what the table contains and when to use it?
   A score of 5 means all tables have excellent, detailed descriptions.
   A score of 1 means most or all tables lack descriptions entirely.

2. **column_descriptions** (1-5) — Do the configured columns have descriptions?
   Are the descriptions clear enough to disambiguate columns with similar names?
   A score of 5 means every configured column has a clear, helpful description.
   A score of 1 means descriptions are mostly absent or uninformative.

3. **overall_completeness** (1-5) — Considering the number of tables and
   the columns configured, is the metadata thorough? Are there obvious gaps
   such as tables with no column configs at all, or important-sounding
   columns with no descriptions?
   A score of 5 means comprehensive coverage with no notable gaps.

Then compute an **overall_metadata_score** as the average of the three factors,
rounded to the nearest integer (minimum 1, maximum 5).

Respond with ONLY this JSON structure:
{{
  "table_descriptions": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "column_descriptions": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "overall_completeness": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "overall_metadata_score": <int 1-5>,
  "tables_missing_descriptions": ["<table identifier>", ...],
  "columns_missing_descriptions": [{{"table": "<table identifier>", "column": "<column_name>"}}  , ...]
}}""")
])

metadata_chain = metadata_prompt | llm | JsonOutputParser()

metadata_result = metadata_chain.invoke({
    "space_name": space.title,
    "space_description": space.description or "No description provided",
    "num_tables": len(tables),
    "metadata": metadata_formatted,
})

metadata_score = metadata_result["overall_metadata_score"]

print(f"Metadata quality score: {metadata_score} / 5")
print(f"  Table descriptions:    {metadata_result['table_descriptions']['score']} / 5 — {metadata_result['table_descriptions']['reasoning']}")
print(f"  Column descriptions:   {metadata_result['column_descriptions']['score']} / 5 — {metadata_result['column_descriptions']['reasoning']}")
print(f"  Overall completeness:  {metadata_result['overall_completeness']['score']} / 5 — {metadata_result['overall_completeness']['reasoning']}")

if metadata_result.get("tables_missing_descriptions"):
    print(f"\n  Tables missing descriptions: {len(metadata_result['tables_missing_descriptions'])}")
    for t in metadata_result["tables_missing_descriptions"]:
        print(f"    - {t}")

if metadata_result.get("columns_missing_descriptions"):
    print(f"\n  Columns missing descriptions: {len(metadata_result['columns_missing_descriptions'])}")
    for c in metadata_result["columns_missing_descriptions"]:
        print(f"    - {c['table']}.{c['column']}")

## Criterion 2 — Benchmark Question Quantity

Score the space based on the number of benchmark questions.
The score scales linearly from **1** (0 questions) to **5** (20+ questions).

$$\text{score} = \min\!\left(5,\; 1 + \frac{\text{count}}{20} \times 4\right)$$

In [ ]:
def score_benchmark_questions(count: int) -> float:
    """
    Score benchmark question coverage on a 1-5 scale.

    - 0 questions  -> 1
    - 20+ questions -> 5
    - Linear interpolation in between
    """
    if count >= 20:
        return 5.0
    return round(1.0 + (count / 20.0) * 4.0, 2)


num_benchmarks = len(benchmark_questions)
benchmark_score = score_benchmark_questions(num_benchmarks)

print(f"Benchmark question count: {num_benchmarks}")
print(f"Benchmark question score: {benchmark_score} / 5")

## Criterion 3 — Benchmark Question Quality

Use the LLM to evaluate the **quality** of the benchmark questions on a 1–5 scale.
The model assesses three factors:

1. **Relevance** — Do the questions align with the space's purpose and reference the available data sources?
2. **Complexity** — Are the questions substantive enough to exercise real analytical workflows, or are they trivially simple?
3. **Uniqueness** — Are the questions distinct from one another, or are there duplicates / near-duplicates?

The chain returns a structured JSON object with the overall quality score and
per-factor breakdowns so the result is machine-readable.

In [ ]:
# --- build a compact description of the available data sources ---
data_sources_description = "\n".join(
    "  • {ident}{desc}{cols}".format(
        ident=tbl["identifier"],
        desc=" — " + " ".join(tbl["description"]) if tbl.get("description") else "",
        cols=" (columns: " + ", ".join(
            c["column_name"] for c in tbl.get("column_configs", [])
        ) + ")" if tbl.get("column_configs") else "",
    )
    for tbl in tables
) if tables else "  (none)"

questions_for_quality = "\n".join(
    f"  {i}. {' '.join(q.get('question', []))}"
    for i, q in enumerate(benchmark_questions, 1)
) if benchmark_questions else "  (none)"

quality_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert evaluator of Databricks AI/BI Genie spaces. "
     "You will be given the space's purpose, its data sources, and its "
     "benchmark questions. Your job is to score the QUALITY of the "
     "benchmark questions on a 1-5 integer scale.\n\n"
     "You MUST respond with valid JSON only — no markdown fences, no "
     "extra text outside the JSON object."),
    ("human", """Score the quality of the benchmark questions for this Genie space.

**Space Name:** {space_name}
**Space Description:** {space_description}

**Available Data Sources:**
{data_sources}

**Benchmark Questions ({num_benchmarks} total):**
{questions_list}

Evaluate the questions on three factors, each scored 1-5:

1. **relevance** (1-5) — How well do the questions relate to the space's
   stated purpose and reference the tables / columns available in the data sources?
   A score of 5 means every question is clearly answerable from the listed tables
   and closely tied to the space's purpose.

2. **complexity** (1-5) — Are the questions analytically substantive?
   Questions that require aggregations, comparisons, filters, or multi-table
   reasoning score higher. Trivially simple questions like "show all rows"
   or single-column lookups score lower.

3. **uniqueness** (1-5) — Are the questions distinct from one another?
   A score of 5 means no duplicates or near-duplicates. Deduct points for
   every pair of questions that ask essentially the same thing.

Then compute an **overall_quality_score** as the average of the three factors,
rounded to the nearest integer (minimum 1, maximum 5).

Respond with ONLY this JSON structure:
{{
  "relevance": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "complexity": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "uniqueness": {{"score": <int 1-5>, "reasoning": "<1-2 sentences>"}},
  "overall_quality_score": <int 1-5>,
  "duplicate_pairs": ["<q_i vs q_j description>", ...],
  "simplistic_questions": ["<question text>", ...]
}}""")
])

quality_chain = quality_prompt | llm | JsonOutputParser()

quality_result = quality_chain.invoke({
    "space_name": space.title,
    "space_description": space.description or "No description provided",
    "data_sources": data_sources_description,
    "num_benchmarks": num_benchmarks,
    "questions_list": questions_for_quality,
})

quality_score = quality_result["overall_quality_score"]

print(f"Benchmark quality score: {quality_score} / 5")
print(f"  Relevance:  {quality_result['relevance']['score']} / 5 — {quality_result['relevance']['reasoning']}")
print(f"  Complexity: {quality_result['complexity']['score']} / 5 — {quality_result['complexity']['reasoning']}")
print(f"  Uniqueness: {quality_result['uniqueness']['score']} / 5 — {quality_result['uniqueness']['reasoning']}")

if quality_result.get("duplicate_pairs"):
    print(f"\n  Duplicate pairs flagged: {len(quality_result['duplicate_pairs'])}")
    for pair in quality_result["duplicate_pairs"]:
        print(f"    - {pair}")

if quality_result.get("simplistic_questions"):
    print(f"\n  Overly simplistic questions flagged: {len(quality_result['simplistic_questions'])}")
    for sq in quality_result["simplistic_questions"]:
        print(f"    - {sq}")

## Overall Evaluation Report

Build a final LangChain evaluation chain that synthesises all computed scores
into a single narrative report with actionable recommendations.

```
prompt  ─►  ChatModel  ─►  StrOutputParser
```

In [ ]:
overall_score = round((metadata_score + benchmark_score + quality_score) / 3, 2)

evaluation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert evaluator of Databricks AI/BI Genie spaces. "
     "Your role is to synthesise multiple evaluation criteria into a "
     "single, clear report with actionable recommendations. "
     "Be concise and specific."),
    ("human", """Produce an overall evaluation report for the following Genie space.

**Space Name:** {space_name}
**Space Description:** {space_description}

**Available Data Sources:**
{data_sources}

**Benchmark Questions ({num_benchmarks} total):**
{questions_list}

---

### Scores

| # | Criterion | Score |
|---|-----------|-------|
| 1 | Table & Column Metadata | {metadata_score} / 5 |
| 2 | Benchmark Quantity | {benchmark_score} / 5 |
| 3 | Benchmark Quality  | {quality_score} / 5 |
|   | **Overall**        | **{overall_score} / 5** |

**Metadata breakdown:**
- Table descriptions:   {meta_table_score} / 5 — {meta_table_reasoning}
- Column descriptions:  {meta_col_score} / 5 — {meta_col_reasoning}
- Overall completeness: {meta_comp_score} / 5 — {meta_comp_reasoning}

{metadata_gaps_info}

**Benchmark quality breakdown:**
- Relevance:  {relevance_score} / 5 — {relevance_reasoning}
- Complexity: {complexity_score} / 5 — {complexity_reasoning}
- Uniqueness: {uniqueness_score} / 5 — {uniqueness_reasoning}

{duplicate_info}
{simplistic_info}

---

Provide your evaluation as a structured report with:
1. **Score Summary** — restate each criterion score and the overall score
2. **Assessment** — evaluate metadata quality, benchmark quantity, and benchmark quality
3. **Recommendations** — 3-5 specific, actionable steps to improve (prioritised by impact)
4. **Priority** — HIGH / MEDIUM / LOW based on how critical the gaps are""")
])

evaluation_chain = evaluation_prompt | llm | StrOutputParser()

In [ ]:
metadata_gaps_info = ""
gaps = []
if metadata_result.get("tables_missing_descriptions"):
    gaps.append("Tables missing descriptions: " + ", ".join(metadata_result["tables_missing_descriptions"]))
if metadata_result.get("columns_missing_descriptions"):
    gaps.append("Columns missing descriptions: " + ", ".join(
        f"{c['table']}.{c['column']}" for c in metadata_result["columns_missing_descriptions"]
    ))
if gaps:
    metadata_gaps_info = "**Metadata gaps flagged:**\n" + "\n".join(f"  - {g}" for g in gaps)

duplicate_info = ""
if quality_result.get("duplicate_pairs"):
    duplicate_info = "**Duplicate pairs flagged:**\n" + "\n".join(
        f"  - {p}" for p in quality_result["duplicate_pairs"]
    )

simplistic_info = ""
if quality_result.get("simplistic_questions"):
    simplistic_info = "**Overly simplistic questions flagged:**\n" + "\n".join(
        f"  - {q}" for q in quality_result["simplistic_questions"]
    )

evaluation_report = evaluation_chain.invoke({
    "space_name": space.title,
    "space_description": space.description or "No description provided",
    "data_sources": data_sources_description,
    "num_benchmarks": num_benchmarks,
    "questions_list": questions_for_quality,
    "metadata_score": metadata_score,
    "meta_table_score": metadata_result["table_descriptions"]["score"],
    "meta_table_reasoning": metadata_result["table_descriptions"]["reasoning"],
    "meta_col_score": metadata_result["column_descriptions"]["score"],
    "meta_col_reasoning": metadata_result["column_descriptions"]["reasoning"],
    "meta_comp_score": metadata_result["overall_completeness"]["score"],
    "meta_comp_reasoning": metadata_result["overall_completeness"]["reasoning"],
    "metadata_gaps_info": metadata_gaps_info,
    "benchmark_score": benchmark_score,
    "quality_score": quality_score,
    "overall_score": overall_score,
    "relevance_score": quality_result["relevance"]["score"],
    "relevance_reasoning": quality_result["relevance"]["reasoning"],
    "complexity_score": quality_result["complexity"]["score"],
    "complexity_reasoning": quality_result["complexity"]["reasoning"],
    "uniqueness_score": quality_result["uniqueness"]["score"],
    "uniqueness_reasoning": quality_result["uniqueness"]["reasoning"],
    "duplicate_info": duplicate_info,
    "simplistic_info": simplistic_info,
})

print(evaluation_report)

## Evaluation Summary

Structured output of all computed scores for downstream consumption.

In [ ]:
summary = {
    "space_id": GENIE_SPACE_ID,
    "space_name": space.title,
    "criteria": {
        "table_column_metadata": {
            "score": metadata_score,
            "max_score": 5,
            "table_descriptions": metadata_result["table_descriptions"],
            "column_descriptions": metadata_result["column_descriptions"],
            "overall_completeness": metadata_result["overall_completeness"],
            "tables_missing_descriptions": metadata_result.get("tables_missing_descriptions", []),
            "columns_missing_descriptions": metadata_result.get("columns_missing_descriptions", []),
        },
        "benchmark_quantity": {
            "count": num_benchmarks,
            "score": benchmark_score,
            "max_score": 5,
            "threshold_for_max": 20,
        },
        "benchmark_quality": {
            "score": quality_score,
            "max_score": 5,
            "relevance": quality_result["relevance"],
            "complexity": quality_result["complexity"],
            "uniqueness": quality_result["uniqueness"],
            "duplicate_pairs": quality_result.get("duplicate_pairs", []),
            "simplistic_questions": quality_result.get("simplistic_questions", []),
        },
    },
    "overall_score": overall_score,
}

print(json.dumps(summary, indent=2))